In [ ]:
import pandas as pd
import pyodbc

# --- Passo 1: Ler o arquivo CSV original com Pandas ---
# IMPORTANTE: Altere o caminho abaixo para o local do seu arquivo.
caminho_csv = r'Datasets/BankChurners.csv'
df = pd.read_csv(caminho_csv)

print("Arquivo CSV carregado com sucesso.")

# --- Passo 2: Renomear as colunas no DataFrame para corresponder à tabela SQL ---
# Esta etapa é CRUCIAL para alinhar os nomes longos do CSV com os nomes curtos do SQL.
# VERIFIQUE se os nomes longos abaixo correspondem exatamente ao seu CSV.
df.rename(columns={
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1': 'Naive_Bayes_Classifier_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2': 'Naive_Bayes_Classifier_2'
}, inplace=True)

print("Colunas do DataFrame renomeadas para corresponder ao SQL.")

# --- Passo 3: Conectar ao Banco e Inserir os Dados ---
# IMPORTANTE: Altere o NOME_DO_SEU_SERVIDOR
conexao_str = (
    r"Driver={ODBC Driver 17 for SQL Server};"
    r"Server=JHON\SQLEXPRESS;" 
    r"Database=Banco_Churn;"
    r"Trusted_Connection=yes;"
)

# Bloco de código para executar a inserção de forma segura
try:
    conexao_sql = pyodbc.connect(conexao_str)
    cursor = conexao_sql.cursor()
    print("Conexão com SQL Server bem-sucedida.")

    # (Opcional, mas recomendado) Limpa a tabela antes de inserir para evitar duplicatas.
    cursor.execute("TRUNCATE TABLE dbo.BankChurners")
    
    # Loop para inserir cada linha do DataFrame na tabela SQL
    for index, row in df.iterrows():
        # A ordem das colunas no INSERT deve ser a mesma da sua tabela
        cursor.execute("""
            INSERT INTO dbo.BankChurners (
                CLIENTNUM, Attrition_Flag, Customer_Age, Gender, Dependent_count,
                Education_Level, Marital_Status, Income_Category, Card_Category,
                Months_on_book, Total_Relationship_Count, Months_Inactive_12_mon,
                Contacts_Count_12_mon, Credit_Limit, Total_Revolving_Bal,
                Avg_Open_To_Buy, Total_Amt_Chng_Q4_Q1, Total_Trans_Amt,
                Total_Trans_Ct, Total_Ct_Chng_Q4_Q1, Avg_Utilization_Ratio,
                Naive_Bayes_Classifier_1, Naive_Bayes_Classifier_2
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            *row.values 
        )
    
    conexao_sql.commit()
    print("Dados inseridos com sucesso no banco de dados!")

except Exception as e:
    print(f"Ocorreu um erro durante a migração: {e}")

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'conexao_sql' in locals():
        conexao_sql.close()
    print("Conexão com o banco de dados fechada.")

Arquivo CSV carregado com sucesso.
Colunas do DataFrame renomeadas para corresponder ao SQL.
Conexão com SQL Server bem-sucedida.
Dados inseridos com sucesso no banco de dados!
Conexão com o banco de dados fechada.
